In [33]:
# Install plotly for interactive charts
!pip install plotly -q

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from ipywidgets import interact
import ipywidgets as widgets
from IPython.display import display, HTML

In [34]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
# Load both datasets
race_results = pd.read_csv('RaceResults.csv')
lt = pd.read_csv('LapTimes.csv')
rr = race_results

print("Race Results:", rr.shape)
print("Lap Times:", lt.shape)

Race Results: (479, 27)
Lap Times: (26692, 35)


In [36]:
# Helper to convert timedelta strings to seconds
def parse_td(s):
    try:
        return pd.to_timedelta(s).total_seconds()
    except:
        return np.nan

# Sort by driver and round
df = rr.sort_values(['Abbreviation', 'Round']).copy()

# FEATURE 1: Recent form — average finish over last 3 races
df['RecentForm'] = df.groupby('Abbreviation')['Position'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
)

# FEATURE 2: Team strength — how many points does this team average per race
team_strength = rr.groupby('TeamName')['Points'].mean().reset_index()
team_strength.columns = ['TeamName', 'TeamStrength']
df = df.merge(team_strength, on='TeamName')

# FEATURE 3: Starting tyre compound
start_tyre = lt[lt['LapNumber'] == 1][['Driver', 'Round', 'Compound']].dropna()
start_tyre.columns = ['Abbreviation', 'Round', 'StartTyre']
df = df.merge(start_tyre, on=['Abbreviation', 'Round'], how='left')

# FEATURE 4: Qualifying time in seconds
df['Q3Secs'] = df['Q3'].apply(parse_td)
df['Q1Secs'] = df['Q1'].apply(parse_td)
df['QualiSecs'] = df['Q3Secs'].fillna(df['Q1Secs'])

# Fill any missing values
df['StartTyre'] = df['StartTyre'].fillna('MEDIUM')
le = LabelEncoder()
df['TyreEnc'] = le.fit_transform(df['StartTyre'])
df['RecentForm'] = df['RecentForm'].fillna(df['GridPosition'])
df['QualiSecs'] = df['QualiSecs'].fillna(df['QualiSecs'].median())

# Features we use to predict
features = ['GridPosition', 'RecentForm', 'TeamStrength', 'TyreEnc', 'QualiSecs']

# Walk-forward prediction — for each race, train only on PAST races
# This simulates a real prediction made before the race
model = RandomForestRegressor(n_estimators=100, random_state=42)
all_preds = []

for round_num in range(1, 25):
    train_data = df[df['Round'] < round_num]
    test_data = df[df['Round'] == round_num].copy()

    if len(train_data) < 20:
        # Not enough history yet — use grid position as prediction
        test_data['PredictedPosition'] = test_data['GridPosition']
    else:
        model.fit(train_data[features], train_data['Position'])
        test_data['PredictedPosition'] = model.predict(test_data[features]).round(1)

    all_preds.append(test_data)

# Combine all predictions
full = pd.concat(all_preds).reset_index(drop=True)
full['ActualPosition'] = full['Position'].astype(int)
full['PredictionError'] = (full['PredictedPosition'] - full['ActualPosition']).round(1)
full['PositionGain'] = (full['GridPosition'] - full['ActualPosition']).round(1)
full['ScoredPoints'] = (full['ActualPosition'] <= 10).astype(int)

# Season summary per driver
season_stats = full.groupby('Abbreviation').agg(
    TotalPoints=('Points', 'sum'),
    Wins=('ActualPosition', lambda x: (x == 1).sum()),
    Podiums=('ActualPosition', lambda x: (x <= 3).sum()),
    PointsFinishes=('ScoredPoints', 'sum'),
    AvgFinish=('ActualPosition', 'mean'),
    AvgGrid=('GridPosition', 'mean'),
    AvgPredError=('PredictionError', lambda x: abs(x).mean()),
    TeamName=('TeamName', 'first')
).round(2).reset_index()

# Feature importance from final trained model
model.fit(df[features], df['Position'])
importance = dict(zip(features, (model.feature_importances_ * 100).round(1)))

print("Model ready!")
print(f"\nFeature importance (what matters most to win):")
for k, v in sorted(importance.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v}%")

Model ready!

Feature importance (what matters most to win):
  GridPosition: 55.8%
  QualiSecs: 17.4%
  RecentForm: 13.7%
  TeamStrength: 9.6%
  TyreEnc: 3.4%


In [37]:
# Official F1 team colors
TEAM_COLORS = {
    'McLaren': '#FF8000',
    'Red Bull Racing': '#3671C6',
    'Mercedes': '#27F4D2',
    'Ferrari': '#E8002D',
    'Aston Martin': '#358C75',
    'Williams': '#64C4FF',
    'Alpine': '#FF87BC',
    'Racing Bulls': '#6692FF',
    'Haas F1 Team': '#B6BABD',
    'Kick Sauber': '#52E252'
}

# Map each driver to their team color
driver_team = rr[['Abbreviation', 'TeamName']].drop_duplicates().set_index('Abbreviation')['TeamName'].to_dict()
driver_color = {d: TEAM_COLORS.get(t, '#888888') for d, t in driver_team.items()}

# Sorted driver list by total points
drivers_sorted = season_stats.sort_values('TotalPoints', ascending=False)['Abbreviation'].tolist()

print("Colors ready for", len(driver_color), "drivers")

Colors ready for 21 drivers


In [38]:
display(HTML("""
<div style="
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%);
    padding: 30px 40px;
    border-radius: 16px;
    margin-bottom: 20px;
    font-family: Arial, sans-serif;
">
    <h1 style="color: #e94560; margin: 0; font-size: 32px; letter-spacing: 2px;">
        &#9654; 2025 F1 SEASON ANALYSIS
    </h1>
    <p style="color: #a8a8b3; margin: 8px 0 0; font-size: 15px;">
        Can we predict who wins before the race starts? — A Machine Learning Story
    </p>
    <div style="display:flex; gap:30px; margin-top:20px;">
        <div style="color:white;">
            <span style="color:#e94560; font-size:24px; font-weight:bold;">24</span>
            <span style="color:#a8a8b3; font-size:12px; display:block;">RACES</span>
        </div>
        <div style="color:white;">
            <span style="color:#e94560; font-size:24px; font-weight:bold;">21</span>
            <span style="color:#a8a8b3; font-size:12px; display:block;">DRIVERS</span>
        </div>
        <div style="color:white;">
            <span style="color:#e94560; font-size:24px; font-weight:bold;">479</span>
            <span style="color:#a8a8b3; font-size:12px; display:block;">RACE ENTRIES</span>
        </div>
        <div style="color:white;">
            <span style="color:#e94560; font-size:24px; font-weight:bold;">26,692</span>
            <span style="color:#a8a8b3; font-size:12px; display:block;">LAPS ANALYZED</span>
        </div>
    </div>
</div>
"""))

In [39]:
# Points race across the season for top 8 drivers
top8 = season_stats.sort_values('TotalPoints', ascending=False).head(8)['Abbreviation'].tolist()

pts = full[full['Abbreviation'].isin(top8)].sort_values(['Abbreviation', 'Round'])
pts['CumPoints'] = pts.groupby('Abbreviation')['Points'].cumsum()

race_labels = rr[['Round', 'Location']].drop_duplicates().sort_values('Round')['Location'].tolist()

fig = go.Figure()

for driver in top8:
    d = pts[pts['Abbreviation'] == driver]
    color = driver_color.get(driver, '#888888')
    fig.add_trace(go.Scatter(
        x=d['Round'],
        y=d['CumPoints'],
        mode='lines+markers',
        name=driver,
        line=dict(color=color, width=2.5),
        marker=dict(size=6, color=color),
        hovertemplate=f'<b>{driver}</b><br>Round %{{x}}<br>Points: %{{y}}<extra></extra>'
    ))

fig.update_layout(
    title=dict(text='Championship Battle — Who Led and When?', font=dict(size=18, color='white')),
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='#16213e',
    font=dict(color='white'),
    xaxis=dict(
        title='Round',
        tickvals=list(range(1, 25)),
        ticktext=[l[:3] for l in race_labels],
        tickfont=dict(size=10),
        gridcolor='#2a2a4a'
    ),
    yaxis=dict(title='Cumulative Points', gridcolor='#2a2a4a'),
    legend=dict(bgcolor='#1a1a2e', bordercolor='#444'),
    height=420
)

fig.show()

In [40]:
# Feature importance bar chart
labels = ['Grid Position', 'Qualifying Time', 'Recent Form', 'Team Strength', 'Tyre Compound']
values = [importance['GridPosition'], importance['QualiSecs'],
          importance['RecentForm'], importance['TeamStrength'], importance['TyreEnc']]
colors_fi = ['#e94560', '#FF8000', '#27F4D2', '#3671C6', '#52E252']

fig2 = go.Figure(go.Bar(
    x=values,
    y=labels,
    orientation='h',
    marker=dict(color=colors_fi),
    text=[f'{v}%' for v in values],
    textposition='outside',
    hovertemplate='%{y}: %{x}%<extra></extra>'
))

fig2.update_layout(
    title=dict(text='What Actually Decides an F1 Race? (Model Feature Importance)', font=dict(size=18, color='white')),
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='#16213e',
    font=dict(color='white'),
    xaxis=dict(title='Importance (%)', gridcolor='#2a2a4a', range=[0, max(values) + 10]),
    yaxis=dict(gridcolor='#2a2a4a'),
    height=350
)

fig2.show()

In [41]:
# Pick a driver here manually — change this to any driver code you want
# Options: NOR, VER, PIA, RUS, LEC, HAM, ALO, SAI, ALB, ANT, etc.
SELECTED_DRIVER = 'NOR'

drv = SELECTED_DRIVER
color = driver_color.get(drv, '#888888')
drv_full = full[full['Abbreviation'] == drv].sort_values('Round')
stats = season_stats[season_stats['Abbreviation'] == drv].iloc[0]
team = driver_team.get(drv, '')

# ---- HEADER ----
display(HTML(f"""
<div style="background:#1a1a2e; border-radius:12px; padding:20px; margin-bottom:15px; font-family:Arial;">
    <div style="display:flex; align-items:center; gap:15px; margin-bottom:15px;">
        <div style="width:50px; height:50px; border-radius:50%; background:{color}22;
                    border:3px solid {color}; display:flex; align-items:center;
                    justify-content:center; font-weight:bold; color:{color}; font-size:14px;">
            {drv}
        </div>
        <div>
            <div style="color:white; font-size:20px; font-weight:bold;">{drv_full['FullName'].iloc[0]}</div>
            <div style="color:#a8a8b3; font-size:13px;">{team}</div>
        </div>
    </div>
    <div style="display:grid; grid-template-columns:repeat(5,1fr); gap:10px;">
        <div style="background:#16213e; border-radius:8px; padding:12px; text-align:center;">
            <div style="color:{color}; font-size:22px; font-weight:bold;">{int(stats.TotalPoints)}</div>
            <div style="color:#a8a8b3; font-size:11px;">POINTS</div>
        </div>
        <div style="background:#16213e; border-radius:8px; padding:12px; text-align:center;">
            <div style="color:{color}; font-size:22px; font-weight:bold;">{int(stats.Wins)}</div>
            <div style="color:#a8a8b3; font-size:11px;">WINS</div>
        </div>
        <div style="background:#16213e; border-radius:8px; padding:12px; text-align:center;">
            <div style="color:{color}; font-size:22px; font-weight:bold;">{int(stats.Podiums)}</div>
            <div style="color:#a8a8b3; font-size:11px;">PODIUMS</div>
        </div>
        <div style="background:#16213e; border-radius:8px; padding:12px; text-align:center;">
            <div style="color:{color}; font-size:22px; font-weight:bold;">{int(stats.PointsFinishes)}</div>
            <div style="color:#a8a8b3; font-size:11px;">POINTS FINISHES</div>
        </div>
        <div style="background:#16213e; border-radius:8px; padding:12px; text-align:center;">
            <div style="color:{color}; font-size:22px; font-weight:bold;">{stats.AvgPredError}</div>
            <div style="color:#a8a8b3; font-size:11px;">AVG PRED ERROR</div>
        </div>
    </div>
</div>
"""))

# ---- CHARTS ----
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Predicted vs Actual Finish', 'Positions Gained/Lost per Race'),
                    horizontal_spacing=0.12)

fig.add_trace(go.Scatter(
    x=drv_full['Round'], y=drv_full['GridPosition'],
    mode='lines+markers', name='Grid',
    line=dict(color='#a8a8b3', dash='dash', width=1.5),
    marker=dict(size=5),
    hovertemplate='R%{x} Grid: %{y}<extra></extra>'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=drv_full['Round'], y=drv_full['PredictedPosition'],
    mode='lines+markers', name='Predicted',
    line=dict(color='#FFD700', width=2),
    marker=dict(size=5, symbol='diamond'),
    hovertemplate='R%{x} Predicted: %{y}<extra></extra>'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=drv_full['Round'], y=drv_full['ActualPosition'],
    mode='lines+markers', name='Actual',
    line=dict(color=color, width=2.5),
    marker=dict(size=7),
    hovertemplate='R%{x} Actual: %{y}<extra></extra>'
), row=1, col=1)

gain_colors = ['#2ecc71' if g > 0 else '#e74c3c' if g < 0 else '#888'
               for g in drv_full['PositionGain']]
fig.add_trace(go.Bar(
    x=drv_full['Round'], y=drv_full['PositionGain'],
    name='Pos Gained', marker_color=gain_colors,
    hovertemplate='R%{x}<br>%{y} positions<extra></extra>'
), row=1, col=2)

fig.add_hline(y=0, line_color='white', line_width=0.8, row=1, col=2)

fig.update_layout(
    paper_bgcolor='#1a1a2e', plot_bgcolor='#16213e',
    font=dict(color='white'), height=400,
    legend=dict(bgcolor='#1a1a2e')
)
fig.update_yaxes(autorange='reversed', gridcolor='#2a2a4a', row=1, col=1)
fig.update_yaxes(gridcolor='#2a2a4a', row=1, col=2)
fig.update_xaxes(gridcolor='#2a2a4a')
fig.show()

# ---- TABLE ----
tyre_colors = {'SOFT': '#e74c3c', 'MEDIUM': '#f39c12', 'HARD': '#95a5a6', 'INTERMEDIATE': '#27ae60'}

rows_html = ''
for _, row in drv_full.iterrows():
    gain = row['PositionGain']
    gain_str = f'+{int(gain)}' if gain > 0 else str(int(gain))
    gain_color = '#2ecc71' if gain > 0 else '#e74c3c' if gain < 0 else '#888'
    pos = int(row['ActualPosition'])
    pos_bg = '#FFD700' if pos == 1 else '#2ecc71' if pos <= 3 else '#3498db' if pos <= 10 else '#555'
    pos_color = '#333' if pos == 1 else '#fff'
    tyre = row['StartTyre'] if pd.notna(row['StartTyre']) else '—'
    tc = tyre_colors.get(tyre, '#888')
    status = row['Status']
    status_color = '#e74c3c' if status == 'Retired' else '#2ecc71'
    pred_err = row['PredictionError']
    err_color = '#2ecc71' if abs(pred_err) <= 2 else '#f39c12' if abs(pred_err) <= 5 else '#e74c3c'
    err_str = f'+{pred_err}' if pred_err > 0 else str(pred_err)

    rows_html += f"""
    <tr style="border-bottom:1px solid #2a2a4a;">
        <td style="padding:8px 12px; color:#a8a8b3;">R{int(row['Round'])}</td>
        <td style="padding:8px 12px; color:white;">{row['Location']}</td>
        <td style="padding:8px 12px; color:#a8a8b3;">{int(row['GridPosition'])}</td>
        <td style="padding:8px 12px;">
            <span style="background:{pos_bg}; color:{pos_color}; padding:2px 8px; border-radius:12px; font-weight:bold; font-size:12px;">P{pos}</span>
        </td>
        <td style="padding:8px 12px; color:#FFD700;">{row['PredictedPosition']}</td>
        <td style="padding:8px 12px; color:{err_color};">{err_str}</td>
        <td style="padding:8px 12px; color:{gain_color}; font-weight:bold;">{gain_str}</td>
        <td style="padding:8px 12px;">
            <span style="background:{tc}22; color:{tc}; padding:2px 8px; border-radius:8px; font-size:11px; border:1px solid {tc}44;">{tyre}</span>
        </td>
        <td style="padding:8px 12px; color:{status_color}; font-size:12px;">{status}</td>
        <td style="padding:8px 12px; color:white; font-weight:bold;">{int(row['Points'])}</td>
    </tr>"""

display(HTML(f"""
<div style="background:#1a1a2e; border-radius:12px; padding:20px; font-family:Arial; overflow-x:auto;">
    <h3 style="color:white; margin:0 0 15px;">Race by Race — {drv_full['FullName'].iloc[0]}</h3>
    <table style="width:100%; border-collapse:collapse; font-size:13px;">
        <thead>
            <tr style="border-bottom:2px solid {color};">
                <th style="padding:8px 12px; color:#a8a8b3; text-align:left;">RND</th>
                <th style="padding:8px 12px; color:#a8a8b3; text-align:left;">RACE</th>
                <th style="padding:8px 12px; color:#a8a8b3; text-align:left;">GRID</th>
                <th style="padding:8px 12px; color:#a8a8b3; text-align:left;">FINISH</th>
                <th style="padding:8px 12px; color:#FFD700; text-align:left;">PREDICTED</th>
                <th style="padding:8px 12px; color:#a8a8b3; text-align:left;">ERROR</th>
                <th style="padding:8px 12px; color:#a8a8b3; text-align:left;">+/- POS</th>
                <th style="padding:8px 12px; color:#a8a8b3; text-align:left;">TYRE</th>
                <th style="padding:8px 12px; color:#a8a8b3; text-align:left;">STATUS</th>
                <th style="padding:8px 12px; color:#a8a8b3; text-align:left;">PTS</th>
            </tr>
        </thead>
        <tbody>{rows_html}</tbody>
    </table>
</div>
"""))

RND,RACE,GRID,FINISH,PREDICTED,ERROR,+/- POS,TYRE,STATUS,PTS
R1,Melbourne,1,P1,1.0,0.0,0,INTERMEDIATE,Finished,25
R2,Shanghai,3,P2,5.3,+3.3,+1,MEDIUM,Finished,18
R3,Suzuka,2,P2,2.8,+0.8,0,MEDIUM,Finished,18
R4,Sakhir,6,P3,6.7,+3.7,+3,SOFT,Finished,15
R5,Jeddah,10,P4,6.4,+2.4,+6,HARD,Finished,12
R6,Miami Gardens,2,P2,2.3,+0.3,0,MEDIUM,Finished,18
R7,Imola,4,P2,6.2,+4.2,+2,MEDIUM,Finished,18
R8,Monaco,1,P1,3.2,+2.2,0,MEDIUM,Finished,25
R9,Barcelona,2,P2,2.7,+0.7,0,SOFT,Finished,18
R10,Montréal,7,P18,4.9,-13.1,-11,HARD,Retired,0


In [42]:
# Biggest overperformers vs model predictions
overperformers = season_stats.copy()
overperformers['OverPerformance'] = (overperformers['AvgGrid'] - overperformers['AvgFinish']).round(2)

fig_op = px.bar(
    overperformers.sort_values('OverPerformance', ascending=True),
    x='OverPerformance',
    y='Abbreviation',
    orientation='h',
    color='TeamName',
    color_discrete_map=TEAM_COLORS,
    title='Who Consistently Beat Expectations? (Avg Grid - Avg Finish)',
    labels={'OverPerformance': 'Positions Gained on Average', 'Abbreviation': 'Driver'},
    text='OverPerformance'
)

fig_op.update_layout(
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='#16213e',
    font=dict(color='white'),
    height=550,
    showlegend=True
)
fig_op.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig_op.show()

In [43]:
from google.colab import output
output.enable_custom_widget_manager()

In [44]:
from ipywidgets import interact
import ipywidgets as widgets

def show_driver(Driver):
    drv = Driver
    color = driver_color.get(drv, '#888888')
    drv_full = full[full['Abbreviation'] == drv].sort_values('Round')
    stats = season_stats[season_stats['Abbreviation'] == drv].iloc[0]
    team = driver_team.get(drv, '')

    display(HTML(f"""
    <div style="background:#1a1a2e; border-radius:12px; padding:20px; margin-bottom:15px; font-family:Arial;">
        <div style="display:flex; align-items:center; gap:15px; margin-bottom:15px;">
            <div style="width:50px; height:50px; border-radius:50%; background:{color}22;
                        border:3px solid {color}; display:flex; align-items:center;
                        justify-content:center; font-weight:bold; color:{color}; font-size:14px;">
                {drv}
            </div>
            <div>
                <div style="color:white; font-size:20px; font-weight:bold;">{drv_full['FullName'].iloc[0]}</div>
                <div style="color:#a8a8b3; font-size:13px;">{team}</div>
            </div>
        </div>
        <div style="display:grid; grid-template-columns:repeat(5,1fr); gap:10px;">
            <div style="background:#16213e; border-radius:8px; padding:12px; text-align:center;">
                <div style="color:{color}; font-size:22px; font-weight:bold;">{int(stats.TotalPoints)}</div>
                <div style="color:#a8a8b3; font-size:11px;">POINTS</div>
            </div>
            <div style="background:#16213e; border-radius:8px; padding:12px; text-align:center;">
                <div style="color:{color}; font-size:22px; font-weight:bold;">{int(stats.Wins)}</div>
                <div style="color:#a8a8b3; font-size:11px;">WINS</div>
            </div>
            <div style="background:#16213e; border-radius:8px; padding:12px; text-align:center;">
                <div style="color:{color}; font-size:22px; font-weight:bold;">{int(stats.Podiums)}</div>
                <div style="color:#a8a8b3; font-size:11px;">PODIUMS</div>
            </div>
            <div style="background:#16213e; border-radius:8px; padding:12px; text-align:center;">
                <div style="color:{color}; font-size:22px; font-weight:bold;">{int(stats.PointsFinishes)}</div>
                <div style="color:#a8a8b3; font-size:11px;">POINTS FINISHES</div>
            </div>
            <div style="background:#16213e; border-radius:8px; padding:12px; text-align:center;">
                <div style="color:{color}; font-size:22px; font-weight:bold;">{stats.AvgPredError}</div>
                <div style="color:#a8a8b3; font-size:11px;">AVG PRED ERROR</div>
            </div>
        </div>
    </div>
    """))

    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=('Predicted vs Actual Finish', 'Positions Gained/Lost per Race'),
                        horizontal_spacing=0.12)

    fig.add_trace(go.Scatter(
        x=drv_full['Round'], y=drv_full['GridPosition'],
        mode='lines+markers', name='Grid',
        line=dict(color='#a8a8b3', dash='dash', width=1.5),
        marker=dict(size=5)
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=drv_full['Round'], y=drv_full['PredictedPosition'],
        mode='lines+markers', name='Predicted',
        line=dict(color='#FFD700', width=2),
        marker=dict(size=5, symbol='diamond')
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=drv_full['Round'], y=drv_full['ActualPosition'],
        mode='lines+markers', name='Actual',
        line=dict(color=color, width=2.5),
        marker=dict(size=7)
    ), row=1, col=1)

    gain_colors = ['#2ecc71' if g > 0 else '#e74c3c' if g < 0 else '#888'
                   for g in drv_full['PositionGain']]
    fig.add_trace(go.Bar(
        x=drv_full['Round'], y=drv_full['PositionGain'],
        name='Pos Gained', marker_color=gain_colors
    ), row=1, col=2)

    fig.add_hline(y=0, line_color='white', line_width=0.8, row=1, col=2)

    fig.update_layout(
        paper_bgcolor='#1a1a2e', plot_bgcolor='#16213e',
        font=dict(color='white'), height=400,
        legend=dict(bgcolor='#1a1a2e')
    )
    fig.update_yaxes(autorange='reversed', gridcolor='#2a2a4a', row=1, col=1)
    fig.update_yaxes(gridcolor='#2a2a4a', row=1, col=2)
    fig.update_xaxes(gridcolor='#2a2a4a')
    fig.show()

    tyre_colors = {'SOFT': '#e74c3c', 'MEDIUM': '#f39c12', 'HARD': '#95a5a6', 'INTERMEDIATE': '#27ae60'}
    rows_html = ''
    for _, row in drv_full.iterrows():
        gain = row['PositionGain']
        gain_str = f'+{int(gain)}' if gain > 0 else str(int(gain))
        gain_color = '#2ecc71' if gain > 0 else '#e74c3c' if gain < 0 else '#888'
        pos = int(row['ActualPosition'])
        pos_bg = '#FFD700' if pos == 1 else '#2ecc71' if pos <= 3 else '#3498db' if pos <= 10 else '#555'
        pos_color = '#333' if pos == 1 else '#fff'
        tyre = row['StartTyre'] if pd.notna(row['StartTyre']) else '—'
        tc = tyre_colors.get(tyre, '#888')
        status = row['Status']
        status_color = '#e74c3c' if status == 'Retired' else '#2ecc71'
        pred_err = row['PredictionError']
        err_color = '#2ecc71' if abs(pred_err) <= 2 else '#f39c12' if abs(pred_err) <= 5 else '#e74c3c'
        err_str = f'+{pred_err}' if pred_err > 0 else str(pred_err)

        rows_html += f"""
        <tr style="border-bottom:1px solid #2a2a4a;">
            <td style="padding:8px 12px; color:#a8a8b3;">R{int(row['Round'])}</td>
            <td style="padding:8px 12px; color:white;">{row['Location']}</td>
            <td style="padding:8px 12px; color:#a8a8b3;">{int(row['GridPosition'])}</td>
            <td style="padding:8px 12px;">
                <span style="background:{pos_bg}; color:{pos_color}; padding:2px 8px; border-radius:12px; font-weight:bold; font-size:12px;">P{pos}</span>
            </td>
            <td style="padding:8px 12px; color:#FFD700;">{row['PredictedPosition']}</td>
            <td style="padding:8px 12px; color:{err_color};">{err_str}</td>
            <td style="padding:8px 12px; color:{gain_color}; font-weight:bold;">{gain_str}</td>
            <td style="padding:8px 12px;">
                <span style="background:{tc}22; color:{tc}; padding:2px 8px; border-radius:8px; font-size:11px; border:1px solid {tc}44;">{tyre}</span>
            </td>
            <td style="padding:8px 12px; color:{status_color}; font-size:12px;">{status}</td>
            <td style="padding:8px 12px; color:white; font-weight:bold;">{int(row['Points'])}</td>
        </tr>"""

    display(HTML(f"""
    <div style="background:#1a1a2e; border-radius:12px; padding:20px; font-family:Arial; overflow-x:auto;">
        <h3 style="color:white; margin:0 0 15px;">Race by Race — {drv_full['FullName'].iloc[0]}</h3>
        <table style="width:100%; border-collapse:collapse; font-size:13px;">
            <thead>
                <tr style="border-bottom:2px solid {color};">
                    <th style="padding:8px 12px; color:#a8a8b3; text-align:left;">RND</th>
                    <th style="padding:8px 12px; color:#a8a8b3; text-align:left;">RACE</th>
                    <th style="padding:8px 12px; color:#a8a8b3; text-align:left;">GRID</th>
                    <th style="padding:8px 12px; color:#a8a8b3; text-align:left;">FINISH</th>
                    <th style="padding:8px 12px; color:#FFD700; text-align:left;">PREDICTED</th>
                    <th style="padding:8px 12px; color:#a8a8b3; text-align:left;">ERROR</th>
                    <th style="padding:8px 12px; color:#a8a8b3; text-align:left;">+/- POS</th>
                    <th style="padding:8px 12px; color:#a8a8b3; text-align:left;">TYRE</th>
                    <th style="padding:8px 12px; color:#a8a8b3; text-align:left;">STATUS</th>
                    <th style="padding:8px 12px; color:#a8a8b3; text-align:left;">PTS</th>
                </tr>
            </thead>
            <tbody>{rows_html}</tbody>
        </table>
    </div>
    """))

# This creates the dropdown — all 21 drivers
interact(show_driver, Driver=widgets.Dropdown(
    options=drivers_sorted,
    value='NOR',
    description='Select Driver:'
))

interactive(children=(Dropdown(description='Select Driver:', options=('NOR', 'VER', 'PIA', 'RUS', 'LEC', 'ANT'…

<function __main__.show_driver(Driver)>

In [45]:


import traceback

print("=== CHECKING ALL REQUIRED VARIABLES ===\n")

checks = {
    'full': 'full',
    'rr': 'rr',
    'lt': 'lt',
    'df': 'df',
    'season_stats': 'season_stats',
    'drivers_sorted': 'drivers_sorted',
    'driver_color': 'driver_color',
    'driver_team': 'driver_team',
    'importance': 'importance',
    'TEAM_COLORS': 'TEAM_COLORS',
}

all_ok = True
for label, var in checks.items():
    try:
        val = eval(var)
        print(f"  ✅ {label}: {type(val).__name__}", end='')
        if hasattr(val, '__len__'):
            print(f" (len={len(val)})", end='')
        print()
    except NameError:
        print(f"  ❌ {label}: NOT FOUND — this variable is missing!")
        all_ok = False

print()

# Check columns
print("=== CHECKING KEY COLUMNS ===\n")
for col in ['PredictedPosition','PredictionError','ActualPosition','PositionGain',
            'Points','GridPosition','StartTyre','Status','Round','FullName',
            'Abbreviation','TeamName','Location']:
    if col in full.columns:
        print(f"  ✅ full['{col}']")
    else:
        print(f"  ❌ full['{col}'] MISSING")

print()
print("=== CHECKING HeadshotUrl ===\n")
sample = rr[['Abbreviation','HeadshotUrl']].drop_duplicates('Abbreviation').head(3)
print(sample.to_string())

print()
print("=== TESTING NEURAL NETWORK IMPORT ===\n")
try:
    from sklearn.neural_network import MLPRegressor
    from sklearn.preprocessing import StandardScaler
    print("  ✅ MLPRegressor available")
except Exception as e:
    print(f"  ❌ {e}")

print()
print("=== TESTING BASE64 ===\n")
try:
    import base64
    test = base64.b64encode(b"hello").decode()
    print(f"  ✅ base64 works: {test}")
except Exception as e:
    print(f"  ❌ {e}")

print()
if all_ok:
    print("✅ All variables present — ready to build dashboard!")
else:
    print("❌ Some variables missing — re-run earlier cells first!")

=== CHECKING ALL REQUIRED VARIABLES ===

  ✅ full: DataFrame (len=479)
  ✅ rr: DataFrame (len=479)
  ✅ lt: DataFrame (len=26692)
  ✅ df: DataFrame (len=479)
  ✅ season_stats: DataFrame (len=21)
  ✅ drivers_sorted: list (len=21)
  ✅ driver_color: dict (len=21)
  ✅ driver_team: dict (len=21)
  ✅ importance: dict (len=5)
  ✅ TEAM_COLORS: dict (len=10)

=== CHECKING KEY COLUMNS ===

  ✅ full['PredictedPosition']
  ✅ full['PredictionError']
  ✅ full['ActualPosition']
  ✅ full['PositionGain']
  ✅ full['Points']
  ✅ full['GridPosition']
  ✅ full['StartTyre']
  ✅ full['Status']
  ✅ full['Round']
  ✅ full['FullName']
  ✅ full['Abbreviation']
  ✅ full['TeamName']
  ✅ full['Location']

=== CHECKING HeadshotUrl ===

  Abbreviation                                                                                                                                             HeadshotUrl
0          NOR    https://media.formula1.com/d_driver_fallback_image.png/content/dam/fom-website/drivers/L/LANNOR01_Lan

In [ ]:
# ============================================================
# CELL 13 — ULTIMATE DASHBOARD (bulletproof version)
# ============================================================

import json, base64, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

# ── STEP 1: NEURAL NETWORK ────────────────────────────────────────────────────
print("⚙️  Training Neural Network...")
try:
    from sklearn.neural_network import MLPRegressor
    from sklearn.preprocessing import StandardScaler

    features = ['GridPosition','RecentForm','TeamStrength','TyreEnc','QualiSecs']
    nn_preds_list = []

    for round_num in range(1, 25):
        tr = df[df['Round'] < round_num].copy()
        te = df[df['Round'] == round_num].copy()
        if len(tr) < 20:
            te['NNPred'] = te['GridPosition']
        else:
            sc = StandardScaler()
            Xtr = sc.fit_transform(tr[features])
            Xte = sc.transform(te[features])
            nn = MLPRegressor(hidden_layer_sizes=(64,32,16), activation='relu',
                              max_iter=300, random_state=42,
                              early_stopping=True, validation_fraction=0.15,
                              n_iter_no_change=15, learning_rate_init=0.001)
            nn.fit(Xtr, tr['Position'])
            te['NNPred'] = np.clip(nn.predict(Xte),1,20).round(1)
        nn_preds_list.append(te[['Abbreviation','Round','NNPred']])

    nn_df = pd.concat(nn_preds_list).reset_index(drop=True)
    full2 = full.merge(nn_df, on=['Abbreviation','Round'], how='left')
    full2['NNPred']  = full2['NNPred'].fillna(full2['PredictedPosition'])
    full2['NNError'] = (full2['NNPred'] - full2['ActualPosition']).round(1)
    nn_acc = round((abs(full2['NNError'])<=2).sum()/len(full2)*100,1)
    nn_mae = round(abs(full2['NNError']).mean(),2)
    print(f"   \u2705 NN done \u2014 \xb12 accuracy: {nn_acc}%  MAE: {nn_mae}")
except Exception as e:
    print(f"   \u26a0\ufe0f  NN fallback: {e}")
    full2 = full.copy()
    full2['NNPred']  = full2['PredictedPosition']
    full2['NNError'] = full2['PredictionError']
    nn_acc = round((abs(full2['NNError'])<=2).sum()/len(full2)*100,1)
    nn_mae = round(abs(full2['NNError']).mean(),2)

# ── STEP 2: PIT STOP DATA ─────────────────────────────────────────────────────
print("\u2699\ufe0f  Processing pit stops...")
try:
    lt_sorted = lt.sort_values(['Driver','Round','LapNumber']).copy()
    lt_sorted['PrevStint'] = lt_sorted.groupby(['Driver','Round'])['Stint'].shift(1)
    pit_rows = lt_sorted[(lt_sorted['Stint'] != lt_sorted['PrevStint']) &
                          lt_sorted['PrevStint'].notna()].copy()
    pit_list = []
    for _, row in pit_rows.iterrows():
        drv, rnd, lap = row['Driver'], row['Round'], row['LapNumber']
        d = lt_sorted[(lt_sorted['Driver']==drv)&(lt_sorted['Round']==rnd)]
        pb = d[d['LapNumber'] < lap]['Position'].dropna()
        pa = d[d['LapNumber'] > lap]['Position'].dropna()
        before = float(pb.iloc[-1]) if len(pb) else None
        after  = float(pa.iloc[0])  if len(pa)  else None
        pit_list.append({
            'Driver':drv, 'Round':int(rnd), 'Location':str(row['Location']),
            'PitLap':int(lap), 'Compound':str(row['Compound']),
            'Before':before, 'After':after,
            'Change':(before-after) if before and after else None
        })
    pit_df = pd.DataFrame(pit_list)
    print(f"   \u2705 Pit stops: {len(pit_df)} found")
except Exception as e:
    print(f"   \u26a0\ufe0f  Pit stop error: {e}")
    pit_df = pd.DataFrame(columns=['Driver','Round','Location','PitLap','Compound','Before','After','Change'])

# ── STEP 3: CONSTANTS ────────────────────────────────────────────────────────
rf_acc = round((abs(full2['PredictionError'])<=2).sum()/len(full2)*100,1)
rf_mae = round(abs(full2['PredictionError']).mean(),2)

TYRE_COL = {'SOFT':'#e74c3c','MEDIUM':'#f39c12','HARD':'#c0c0c0','INTERMEDIATE':'#27ae60'}

def hq(url):
    if not isinstance(url, str) or not url.startswith('http'): return ''
    return url.replace('/1col/','/2col-retina/').replace('.transform/1col/','.transform/2col-retina/')

shots    = {r['Abbreviation']:hq(r['HeadshotUrl']) for _,r in rr[['Abbreviation','HeadshotUrl']].drop_duplicates('Abbreviation').iterrows()}
drv_nums = {r['Abbreviation']:int(r['DriverNumber']) for _,r in rr[['Abbreviation','DriverNumber']].drop_duplicates('Abbreviation').iterrows()}

LIVERY = {
    'McLaren':        '<svg viewBox="0 0 52 26"><rect width="52" height="26" fill="#FF8000" rx="3"/><rect y="15" width="52" height="11" fill="#111"/><text x="26" y="12" text-anchor="middle" fill="white" font-size="7.5" font-weight="bold" font-family="Arial">McLAREN</text></svg>',
    'Red Bull Racing':'<svg viewBox="0 0 52 26"><rect width="52" height="26" fill="#1B3FAB" rx="3"/><rect width="26" height="26" fill="#CC1E1E" rx="3"/><text x="39" y="16" text-anchor="middle" fill="white" font-size="6.5" font-weight="bold" font-family="Arial">RBR</text></svg>',
    'Mercedes':       '<svg viewBox="0 0 52 26"><rect width="52" height="26" fill="#00D2BE" rx="3"/><text x="26" y="16" text-anchor="middle" fill="#111" font-size="6" font-weight="bold" font-family="Arial">MERCEDES</text></svg>',
    'Ferrari':        '<svg viewBox="0 0 52 26"><rect width="52" height="26" fill="#E8002D" rx="3"/><rect x="18" y="3" width="16" height="18" fill="#FFD700" rx="2"/><text x="26" y="24" text-anchor="middle" fill="white" font-size="5.5" font-family="Arial" font-weight="bold">FERRARI</text></svg>',
    'Aston Martin':   '<svg viewBox="0 0 52 26"><rect width="52" height="26" fill="#236B4C" rx="3"/><text x="26" y="16" text-anchor="middle" fill="#B4E0CC" font-size="5.5" font-weight="bold" font-family="Arial">ASTON MARTIN</text></svg>',
    'Williams':       '<svg viewBox="0 0 52 26"><rect width="52" height="26" fill="#005AFF" rx="3"/><rect y="0" width="52" height="13" fill="#00A3E0"/><text x="26" y="10" text-anchor="middle" fill="white" font-size="6" font-weight="bold" font-family="Arial">WILLIAMS</text></svg>',
    'Alpine':         '<svg viewBox="0 0 52 26"><rect width="52" height="26" fill="#0090FF" rx="3"/><rect y="15" width="52" height="11" fill="#FF0078"/><text x="26" y="13" text-anchor="middle" fill="white" font-size="7" font-weight="bold" font-family="Arial">ALPINE</text></svg>',
    'Racing Bulls':   '<svg viewBox="0 0 52 26"><rect width="52" height="26" fill="#4467FF" rx="3"/><rect y="17" width="52" height="9" fill="#111"/><text x="26" y="14" text-anchor="middle" fill="white" font-size="5.5" font-weight="bold" font-family="Arial">RACING BULLS</text></svg>',
    'Haas F1 Team':   '<svg viewBox="0 0 52 26"><rect width="52" height="26" fill="#E8E8E8" rx="3"/><rect y="10" width="52" height="6" fill="#CC0000"/><text x="26" y="23" text-anchor="middle" fill="#111" font-size="7" font-weight="bold" font-family="Arial">HAAS</text></svg>',
    'Kick Sauber':    '<svg viewBox="0 0 52 26"><rect width="52" height="26" fill="#0D0D0D" rx="3"/><rect y="11" width="52" height="4" fill="#39B54A"/><text x="26" y="10" text-anchor="middle" fill="#39B54A" font-size="5.5" font-weight="bold" font-family="Arial">KICK SAUBER</text></svg>',
}
def livery_b64(team):
    svg = LIVERY.get(team, '<svg viewBox="0 0 52 26"><rect width="52" height="26" fill="#333" rx="3"/></svg>')
    return 'data:image/svg+xml;base64,' + base64.b64encode(svg.encode()).decode()

_CAR_COLORS = {'McLaren':'#FF8000','Red Bull Racing':'#1B3FAB','Mercedes':'#00D2BE','Ferrari':'#E8002D','Aston Martin':'#236B4C','Williams':'#005AFF','Alpine':'#0090FF','Racing Bulls':'#4467FF','Haas F1 Team':'#CCCCCC','Kick Sauber':'#141414'}
_CAR_ACCENT = {'McLaren':'#111','Red Bull Racing':'#CC1E1E','Mercedes':'#111','Ferrari':'#FFD700','Aston Martin':'#1a5c48','Williams':'#0090d0','Alpine':'#FF0078','Racing Bulls':'#0d0d0d','Haas F1 Team':'#CC0000','Kick Sauber':'#39B54A'}
TEAM_CARS_B64 = {}
for _team, _c in _CAR_COLORS.items():
    _a = _CAR_ACCENT.get(_team, '#111')
    _svg = ('<svg viewBox="0 0 340 110" xmlns="http://www.w3.org/2000/svg">'
        '<ellipse cx="170" cy="100" rx="115" ry="9" fill="#000" opacity=".3"/>'
        '<rect x="58" y="10" width="9" height="30" rx="2" fill="#ffffff" opacity=".65"/>'
        '<rect x="273" y="10" width="9" height="30" rx="2" fill="#ffffff" opacity=".65"/>'
        '<rect x="65" y="11" width="210" height="9" rx="2" fill="#ffffff"/>'
        '<rect x="75" y="18" width="190" height="5" rx="1" fill="'+_c+'" opacity=".55"/>'
        '<path d="M98 32 L80 40 L80 50 L98 50 Z" fill="'+_c+'" opacity=".65"/>'
        '<path d="M242 32 L260 40 L260 50 L242 50 Z" fill="'+_c+'" opacity=".65"/>'
        '<ellipse cx="90" cy="54" rx="23" ry="15" fill="#141414"/>'
        '<ellipse cx="90" cy="54" rx="18" ry="11" fill="#232323"/>'
        '<ellipse cx="90" cy="54" rx="9" ry="5.5" fill="'+_c+'"/>'
        '<ellipse cx="90" cy="54" rx="3.5" ry="2.2" fill="#111"/>'
        '<ellipse cx="250" cy="54" rx="23" ry="15" fill="#141414"/>'
        '<ellipse cx="250" cy="54" rx="18" ry="11" fill="#232323"/>'
        '<ellipse cx="250" cy="54" rx="9" ry="5.5" fill="'+_c+'"/>'
        '<ellipse cx="250" cy="54" rx="3.5" ry="2.2" fill="#111"/>'
        '<path d="M113 32 L103 50 L103 80 L113 86 L227 86 L237 80 L237 50 L227 32 Z" fill="'+_c+'"/>'
        '<path d="M127 32 L118 56 L118 80 L127 82 L213 82 L222 80 L222 56 L213 32 Z" fill="'+_a+'"/>'
        '<rect x="138" y="34" width="64" height="10" rx="2" fill="#050518"/>'
        '<path d="M155 44 Q170 30 185 44" stroke="#ffffff" stroke-width="4" fill="none" stroke-linecap="round" opacity=".9"/>'
        '<rect x="168" y="18" width="4" height="18" rx="1.5" fill="#ffffff" opacity=".9"/>'
        '<path d="M55 68 L40 74 L295 74 L315 68 Z" fill="'+_c+'" opacity=".8"/>'
        '<rect x="295" y="68" width="45" height="7" rx="2" fill="'+_c+'"/>'
        '</svg>')
    TEAM_CARS_B64[_team] = 'data:image/svg+xml;base64,' + base64.b64encode(_svg.encode()).decode()

fi_labels = ['Grid Position','Qualifying Time','Recent Form','Team Strength','Tyre Compound']
fi_keys   = ['GridPosition','QualiSecs','RecentForm','TeamStrength','TyreEnc']
fi_vals   = [round(importance.get(k,0),1) for k in fi_keys]
fi_cols   = ['#e94560','#FF8000','#27F4D2','#3671C6','#52E252']
fi_max    = max(fi_vals) or 1
fi_html   = ''.join([
    f'<div class="fi-r"><span class="fi-l">{l}</span>'
    f'<div class="fi-t"><div class="fi-f" style="width:{round(v/fi_max*100)}%;background:{c};box-shadow:0 0 6px {c}88;"></div></div>'
    f'<span style="color:{c};font-size:11px;font-weight:700;font-family:Orbitron,monospace;width:36px;text-align:right;">{v}%</span></div>'
    for l,v,c in zip(fi_labels,fi_vals,fi_cols)
])

top8 = drivers_sorted[:8]
cum_pts = {}
for _d in drivers_sorted:
    _d2 = full[full['Abbreviation']==_d].sort_values('Round')
    cum_pts[_d] = _d2['Points'].cumsum().tolist()
races_sorted = sorted(full['Round'].unique().tolist())
race_locs    = [full[full['Round']==r]['Location'].iloc[0] for r in races_sorted]
champ_data = json.dumps({d:{'r':races_sorted,'c':cum_pts[d],'col':driver_color[d]} for d in top8})
rl_data    = json.dumps(race_locs)

print("\u2699\ufe0f  Building driver data...")

# ── STEP 4: PER-DRIVER DATA ──────────────────────────────────────────────────
all_data = {}
for drv in drivers_sorted:
    d2      = full2[full2['Abbreviation']==drv].sort_values('Round')
    stats   = season_stats[season_stats['Abbreviation']==drv].iloc[0]
    team    = driver_team.get(drv,'')
    car_img = TEAM_CARS_B64.get(team, '')
    color   = driver_color.get(drv,'#888')
    name    = str(d2['FullName'].iloc[0])
    img     = shots.get(drv,'')
    num     = drv_nums.get(drv, 0)
    liv     = livery_b64(team)
    pts=int(stats.TotalPoints); wins=int(stats.Wins); pods=int(stats.Podiums)
    pf=int(stats.PointsFinishes); af=round(float(stats.AvgFinish),1); ag=round(float(stats.AvgGrid),1)
    rets=int((d2['Status']=='Retired').sum()); bf=int(d2['ActualPosition'].min()); op=round(ag-af,2)
    rf_e=round(float(stats.AvgPredError),1); nn_e=round(float(abs(d2['NNError']).mean()),1)
    rf_a=int(round((abs(d2['PredictionError'])<=2).sum()/len(d2)*100))
    nn_a=int(round((abs(d2['NNError'])<=2).sum()/len(d2)*100))
    fn = name.split()[0]
    if wins>=7:   s = f"{fn} was the defining force of the 2025 season, racking up {wins} wins and {pods} podiums."
    elif wins>=5: s = f"Dominant 2025 campaign for {fn} \u2014 {wins} wins and {pods} podiums driving {team}."
    elif wins>=1: s = f"Strong 2025 for {fn}: {wins} win(s), {pods} podiums. P{ag} avg grid to P{af} avg finish."
    elif pods>=3: s = f"{pods} podiums but zero wins \u2014 consistently fast for {team}. P{af} avg finish."
    elif pts>50:  s = f"Solid midfield: {pts} pts for {team}. Averaged P{af} at the flag."
    else:         s = f"Tough season: {pts} pts, {rets} DNF(s). Averaged P{af} for {team}."
    if op>1.5:    s += f" Marginally stronger on race day, gaining {op:.1f} positions vs grid."
    elif op<-1.5: s += f" Marginally stronger on Saturday, dropping {abs(op):.1f} positions per race."
    if rets>=3:   s += f" {rets} retirements cost valuable points at crucial moments."
    best_m = "Random Forest" if rf_a>=nn_a else "Neural Network"
    best_a = max(rf_a,nn_a)
    s += f" {best_m} predicted {best_a}% of races within \xb12 positions."
    dp_drv = pit_df[pit_df['Driver']==drv]
    prd_list = []
    for _,row in d2.iterrows():
        rnd = int(row['Round']); stops = max(1,len(dp_drv[dp_drv['Round']==rnd]))
        prd_list.append({'r':rnd,'loc':str(row['Location']),'g':int(row['GridPosition']),
            'p':int(row['ActualPosition']),'pts':int(row['Points']),'st':str(row['Status']),
            'sp':stops,'gl':int(row['PositionGain'])})
    prd_json = json.dumps(prd_list)
    chart_json = json.dumps({'rds':d2['Round'].tolist(),'grid':d2['GridPosition'].tolist(),
        'rf':d2['PredictedPosition'].tolist(),'nn':d2['NNPred'].tolist(),
        'act':d2['ActualPosition'].tolist(),'gain':d2['PositionGain'].tolist(),'col':color})
    rows = ''
    for _,row in d2.iterrows():
        g=row['PositionGain']; gs=('+' if g>0 else '')+str(int(g)); gc='#2ecc71' if g>0 else '#e74c3c' if g<0 else '#555'
        p=int(row['ActualPosition'])
        pb='linear-gradient(135deg,#FFD700,#FFA500)' if p==1 else 'linear-gradient(135deg,#e74c3c,#c0392b)' if p<=3 else 'linear-gradient(135deg,#3498db,#2980b9)' if p<=10 else '#1a1a35'
        pc='#111' if p<=3 else '#fff'
        ty=str(row['StartTyre']) if pd.notna(row['StartTyre']) else '\u2014'
        tc=TYRE_COL.get(ty,'#888'); st=str(row['Status'])
        sc2='#e74c3c' if st=='Retired' else '#2ecc71' if st=='Finished' else '#f39c12'
        re_=row['PredictionError']; ne_=row['NNError']
        rc='#2ecc71' if abs(re_)<=2 else '#f39c12' if abs(re_)<=5 else '#e74c3c'
        nc='#2ecc71' if abs(ne_)<=2 else '#f39c12' if abs(ne_)<=5 else '#e74c3c'
        rs=('+' if re_>0 else '')+str(re_); ns=('+' if ne_>0 else '')+str(ne_)
        rows+=(f'<tr class="tr"><td class="tm">R{int(row["Round"])}</td><td class="tw">{row["Location"]}</td>'
            f'<td class="tm">{int(row["GridPosition"])}</td><td><div class="pb" style="background:{pb};color:{pc};">P{p}</div></td>'
            f'<td class="tg">{row["PredictedPosition"]}</td><td style="color:{rc};font-weight:700;">{rs}</td>'
            f'<td style="color:#a78bfa;font-weight:700;">{row["NNPred"]}</td><td style="color:{nc};font-weight:700;">{ns}</td>'
            f'<td style="color:{gc};font-weight:700;">{gs}</td>'
            f'<td><span class="tp" style="color:{tc};border-color:{tc}55;background:{tc}15;">{ty}</span></td>'
            f'<td style="color:{sc2};font-size:11px;">{st}</td><td class="tw" style="font-weight:700;">{int(row["Points"])}</td></tr>')
    op_c=('+' if op>0 else '')+str(op); op_clr='#2ecc71' if op>0 else '#e74c3c'
    nn_c='#2ecc71' if nn_a>=70 else '#f39c12' if nn_a>=50 else '#e74c3c'
    rf_c2='#2ecc71' if rf_a>=70 else '#f39c12' if rf_a>=50 else '#e74c3c'
    stat_tiles=''.join([f'<div class="st"><div class="sv" style="color:{vc};">{vv}</div><div class="sl">{lb}</div></div>'
        for vv,lb,vc in [(pts,'POINTS',color),(wins,'WINS',color),(pods,'PODIUMS',color),
            (pf,'PTS FIN.',color),(af,'AVG FIN.',color),(ag,'AVG GRID',color),
            (op_c,'AVG GAIN',op_clr),(bf,'BEST FIN.',color),
            (rets,'DNFs','#e74c3c' if rets>3 else color),(rf_e,'RF ERR',rf_c2)]])
    all_data[drv] = dict(name=name,team=team,color=color,img=img,num=num,liv=liv,car=car_img,
        pts=pts,wins=wins,pods=pods,pf=pf,af=af,ag=ag,rets=rets,bf=bf,op=op,op_s=op_c,op_c=op_clr,
        rf_e=rf_e,nn_e=nn_e,rf_a=rf_a,nn_a=nn_a,nn_c=nn_c,rf_c=rf_c2,
        best_m=best_m,best_a=best_a,summary=s,rows=rows,stat_tiles=stat_tiles,
        chart_json=chart_json,prd_json=prd_json)

# ── STEP 5: SIDEBAR ──────────────────────────────────────────────────────────
sidebar = ''
for rank,drv in enumerate(drivers_sorted,1):
    d = all_data[drv]
    col = d['color']; img = d['img']; liv = d['liv']; nm = d['name']; pts = d['pts']
    sidebar += (
        f'<button class="dbtn" onclick="go(\'{drv}\')" id="btn-{drv}" style="--tc:{col};">'
        f'<span class="dr">{rank}</span>'
        f'<div class="dav" style="border-color:{col}66;">'
        f'<img src="{img}" alt="{drv}" onerror="this.style.display=\'none\';this.nextSibling.style.display=\'flex\';">'
        f'<span class="dfb" style="color:{col};display:none;">{drv}</span></div>'
        f'<div class="di"><div class="dt"><span class="da" style="color:{col};">{drv}</span>'
        f'<img class="dl" src="{liv}" alt=""></div>'
        f'<span class="dn">{nm}</span></div>'
        f'<span class="dp">{pts}</span></button>'
    )

# ── STEP 6: ACCURACY BARS ────────────────────────────────────────────────────
acc_html = ''
for drv in drivers_sorted:
    d = all_data[drv]
    col = d['color']; rfa = d['rf_a']; nna = d['nn_a']
    acc_html += (
        f'<div class="ar"><span class="ad" style="color:{col};">{drv}</span>'
        f'<div class="ab">'
        f'<div class="abw" title="Random Forest"><div class="af" style="width:{rfa}%;background:{col}55;"></div></div>'
        f'<div class="abw" title="Neural Network"><div class="af" style="width:{nna}%;background:{col};box-shadow:0 0 4px {col}66;"></div></div>'
        f'</div>'
        f'<span style="color:{col};font-family:Orbitron,monospace;font-size:10px;width:32px;text-align:right;">{nna}%</span></div>'
    )

# ── STEP 7: DRIVER PANELS ────────────────────────────────────────────────────
panels = ''
for drv in drivers_sorted:
    d = all_data[drv]
    col      = d['color']
    best_col = d['rf_c'] if d['rf_a'] >= d['nn_a'] else d['nn_c']
    # Pull values into plain variables to avoid backslash-in-f-string issues
    num      = d['num'];    img      = d['img'];    liv      = d['liv']
    car      = d['car'];    nm       = d['name'];   team     = d['team']
    rfc      = d['rf_c'];   nnc      = d['nn_c'];   rfa      = d['rf_a']
    nna      = d['nn_a'];   best_m   = d['best_m']; best_a   = d['best_a']
    tiles    = d['stat_tiles']
    summary  = d['summary']; rows_html = d['rows']
    cj       = d['chart_json']; pj = d['prd_json']

    panels += (
        f'\n<div class="dpanel" id="panel-{drv}" style="--tc:{col};">'
        f'\n  <div class="hero">'
        f'\n    <div class="hstripe"></div>'
        f'\n    <div class="hghost">{num}</div>'
        f'\n    <div class="hphoto-col"><div class="hframe" style="border-color:{col};">'
        f'\n      <img class="himg" src="{img}" alt="{drv}" onerror="this.style.display=\'none\';this.nextSibling.style.display=\'flex\';">'
        f'\n      <span class="hfallback" style="display:none;color:{col};">{drv}</span>'
        f'\n      <div class="hglow" style="background:radial-gradient(ellipse at bottom,{col}55,transparent 68%);"></div>'
        f'\n    </div></div>'
        f'\n    <div class="hinfo">'
        f'\n      <div class="hliv-row"><img class="hliv" src="{liv}" alt=""><span class="hnb" style="color:{col};border-color:{col}44;">#{num}</span></div>'
        f'\n      <div class="hname">{nm.upper()}</div>'
        f'\n      <div class="hteam" style="color:{col};">{team.upper()}</div>'
        f'\n      <div class="hmodels">'
        f'\n        <div class="hmb" style="border-color:{rfc}33;background:{rfc}0d;"><span class="hmv" style="color:{rfc};">{rfa}%</span><span class="hml">RANDOM FOREST</span></div>'
        f'\n        <div class="hmb" style="border-color:{nnc}33;background:{nnc}0d;"><span class="hmv" style="color:{nnc};">{nna}%</span><span class="hml">NEURAL NETWORK</span></div>'
        f'\n      </div>'
        f'\n      <div style="font-size:12px;color:{best_col};letter-spacing:.04em;">&#9650; {best_m} was more accurate &mdash; {best_a}% within &plusmn;2 positions</div>'
        f'\n    </div>'
        f'\n    <div class="hero-car-wrap"><img class="hero-car" src="{car}" alt=""></div>'
        f'\n  </div>'
        f'\n  <div class="sstrip">{tiles}</div>'
        f'\n  <div class="pad">'
        f'\n    <div class="sumcard" style="border-top-color:{col};"><div class="clbl">SEASON ANALYSIS</div><p class="sumtxt">{summary}</p></div>'
        f'\n    <div class="charts2">'
        f'\n      <div class="ccard"><div class="clbl">RF vs NEURAL NET vs ACTUAL FINISH POSITION</div><div id="cp-{drv}" style="height:230px;"></div></div>'
        f'\n      <div class="ccard"><div class="clbl">POSITIONS GAINED / LOST PER RACE</div><div id="cg-{drv}" style="height:230px;"></div></div>'
        f'\n    </div>'
        f'\n    <div class="ccard"><div class="clbl">RACE STRATEGY &amp; CHAMPIONSHIP IMPACT</div><div id="strat-{drv}"></div></div>'
        f'\n    <div class="clbl" style="margin-bottom:8px;">RACE BY RACE \u2014 RANDOM FOREST vs NEURAL NETWORK</div>'
        f'\n    <div class="twrap"><table class="rtbl">'
        f'\n      <thead><tr style="border-bottom:1px solid {col}44;"><th>RND</th><th>RACE</th><th>GRID</th><th>FINISH</th>'
        f'<th class="thrf">RF</th><th class="thrf">RF&plusmn;</th><th class="thnn">NN</th><th class="thnn">NN&plusmn;</th>'
        f'<th>GAIN</th><th>TYRE</th><th>STATUS</th><th>PTS</th></tr></thead>'
        f'\n      <tbody>{rows_html}</tbody>'
        f'\n    </table></div>'
        f'\n  </div>'
        # ── Per-panel inline script ──────────────────────────────────────
        f'\n<script>\n  (function(){{\n'
        f'    var cd={cj};\n'
        f'    var prd={pj};\n'
        f'    var col=\'{col}\';\n'   # FIX: semicolon BEFORE newline
        f'    var base={{paper_bgcolor:\'transparent\',plot_bgcolor:\'transparent\','
        f'font:{{color:\'#555577\',size:10,family:\'Orbitron,monospace\'}},'
        f'margin:{{l:32,r:8,t:8,b:44}},'
        f'xaxis:{{gridcolor:\'#181830\',tickfont:{{size:9}}}},'
        f'hovermode:\'x unified\','
        f'hoverlabel:{{bgcolor:\'#050518\',font:{{color:\'white\',size:11}}}}}};\n'
        f'    var cfg={{responsive:true,displayModeBar:false}};\n'
        f'    Plotly.newPlot(\'cp-{drv}\',['
        f'{{x:cd.rds,y:cd.grid,mode:\'lines\',name:\'Grid\',line:{{color:\'#252545\',dash:\'dot\',width:1.5}},hovertemplate:\'R%{{x}} Grid:%{{y}}<extra></extra>\'}},'
        f'{{x:cd.rds,y:cd.rf,mode:\'lines+markers\',name:\'RF Pred\',line:{{color:\'#FFD700\',width:2,dash:\'dash\'}},marker:{{size:4,symbol:\'diamond\',color:\'#FFD700\'}},hovertemplate:\'R%{{x}} RF:%{{y}}<extra></extra>\'}},'
        f'{{x:cd.rds,y:cd.nn,mode:\'lines+markers\',name:\'Neural Net\',line:{{color:\'#a78bfa\',width:2}},marker:{{size:4,color:\'#a78bfa\'}},hovertemplate:\'R%{{x}} NN:%{{y}}<extra></extra>\'}},'
        f'{{x:cd.rds,y:cd.act,mode:\'lines+markers\',name:\'Actual\',line:{{color:col,width:2.5}},marker:{{size:7,color:col}},fill:\'tonexty\',fillcolor:col+\'15\',hovertemplate:\'R%{{x}} Actual:P%{{y}}<extra></extra>\'}}],'
        f'Object.assign({{}},base,{{yaxis:{{autorange:\'reversed\',gridcolor:\'#181830\',title:\'\'}},legend:{{bgcolor:\'transparent\',orientation:\'h\',y:-0.38,font:{{size:9}}}}}}),cfg);\n'
        f'    Plotly.newPlot(\'cg-{drv}\',['
        f'{{x:cd.rds,y:cd.gain,type:\'bar\',marker:{{color:cd.gain.map(function(g){{return g>0?\'#2ecc71\':(g<0?\'#e74c3c\':\'#333\')}}),line:{{width:0}}}},hovertemplate:\'R%{{x}}: %{{y}} positions<extra></extra>\'}}],'
        f'Object.assign({{}},base,{{yaxis:{{gridcolor:\'#181830\',zeroline:true,zerolinecolor:\'#333\',zerolinewidth:1}}}}),cfg);\n'
        f'    var SC={{\'Finished\':\'#2ecc71\',\'Retired\':\'#e74c3c\',\'Lapped\':\'#f39c12\',\'Disqualified\':\'#e74c3c\',\'Did not start\':\'#888\'}};\n'
        f'    var el=document.getElementById(\'strat-{drv}\');var tot=0,h=\'<div class="strat-scroll">\';\n'
        f'    prd.forEach(function(r){{'
        f'tot+=r.pts;'
        f'var gc=r.gl>0?\'#2ecc71\':(r.gl<0?\'#e74c3c\':\'#888\');'
        f'var gs=r.gl>0?\'+\'+r.gl:\'\'+r.gl;'
        f'var sc=SC[r.st]||\'#888\';'
        f'var pb=r.p===1?\'linear-gradient(135deg,#FFD700,#FFA500)\':(r.p<=3?\'linear-gradient(135deg,#e74c3c,#c0392b)\':(r.p<=10?\'linear-gradient(135deg,#3498db,#2980b9)\':\'#1a1a35\'));'
        f'var pt=r.p<=3?\'#111\':\'#fff\';'
        f'var sp=r.sp+\' stop\'+(r.sp!==1?\'s\':\'\');'
        f'h+=\'<div class="sc"><span class="sc-rnd">R\'+r.r+\'</span><span class="sc-loc">\'+r.loc+\'</span>'
        f'<div class="sc-pos" style="background:\'+pb+\';color:\'+pt+\';">P\'+r.p+\'</div>'
        f'<div class="sc-gain" style="color:\'+gc+\';">\'+gs+\'</div>'
        f'<div class="sc-stops" style="color:\'+sc+\';">\'+sp+\'</div>'
        f'<div class="sc-bot"><span class="sc-pts">\'+r.pts+\'pt</span><span class="sc-cumpts">&#x2211;\'+tot+\'</span></div></div>\';}});\n'
        f'    h+=\'</div>\';el.innerHTML=h;\n'
        f'  }})();\n</script>\n</div>\n'
    )

print("\u2699\ufe0f  Building HTML...")

HTML = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>F1 2025 Intelligence</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<link href="https://fonts.googleapis.com/css2?family=Orbitron:wght@400;700;900&family=Rajdhani:wght@400;600;700&display=swap" rel="stylesheet">
<style>
:root{{--bg:#04040f;--s:#08081a;--c:#0c0c20;--b1:#181830;--b2:#222240;
      --tx:#c0c0e0;--mu:#50506a;--su:#8080a0;--red:#e94560;--gold:#FFD700;}}
*{{box-sizing:border-box;margin:0;padding:0;}}
html,body{{height:100%;overflow:hidden;}}
body{{background:var(--bg);font-family:'Rajdhani',sans-serif;color:var(--tx);display:flex;flex-direction:column;}}
#intro{{position:fixed;inset:0;z-index:9999;background:#000;display:flex;flex-direction:column;align-items:center;justify-content:center;overflow:hidden;}}
.igrid{{position:absolute;inset:0;background-image:linear-gradient(var(--b1) 1px,transparent 1px),linear-gradient(90deg,var(--b1) 1px,transparent 1px);background-size:50px 50px;opacity:.45;}}
.icar{{position:absolute;bottom:74px;left:-260px;animation:drive 2.8s cubic-bezier(.12,.7,.3,1) forwards;}}
@keyframes drive{{0%{{left:-260px;}}50%{{left:calc(50% - 105px);}}78%{{left:calc(50% - 105px);}}100%{{left:112vw;}}}}
.iblur{{position:absolute;bottom:70px;left:-260px;width:210px;height:16px;background:radial-gradient(ellipse,#e9456055,transparent 70%);filter:blur(6px);animation:drive 2.8s cubic-bezier(.12,.7,.3,1) forwards;}}
.isl{{position:absolute;height:1px;opacity:0;background:linear-gradient(90deg,transparent,#e9456088,transparent);animation:slA .5s ease-out forwards;}}
@keyframes slA{{0%{{opacity:0;transform:scaleX(0);transform-origin:left;}}40%{{opacity:.8;transform:scaleX(1);}}100%{{opacity:0;}}}}
.itxt{{display:flex;flex-direction:column;align-items:center;gap:10px;}}
.ilogo{{font-family:'Orbitron',monospace;font-size:62px;font-weight:900;color:white;letter-spacing:.12em;opacity:0;transform:translateY(20px);animation:fu .5s ease 2.1s forwards;}}
.ilogo span{{color:var(--red);}}
.isub{{font-size:12px;color:var(--mu);letter-spacing:.3em;opacity:0;animation:fu .5s ease 2.3s forwards;}}
.iprog{{width:290px;height:2px;background:var(--b2);border-radius:2px;overflow:hidden;opacity:0;animation:fu .3s ease 2.4s forwards;}}
.ibar{{height:100%;background:var(--red);width:0;box-shadow:0 0 10px var(--red);animation:bf .8s ease 2.5s forwards;}}
.icta{{margin-top:16px;background:none;border:1px solid var(--red);color:var(--red);font-family:'Orbitron',monospace;font-size:11px;letter-spacing:.15em;padding:10px 28px;border-radius:4px;cursor:pointer;opacity:0;animation:fu .5s ease 2.9s forwards;transition:all .2s;}}
.icta:hover{{background:var(--red);color:white;box-shadow:0 0 20px #e9456044;}}
@keyframes fu{{to{{opacity:1;transform:none;}}}}
@keyframes bf{{to{{width:100%;}}}}
#app{{display:flex;flex:1;overflow:hidden;opacity:0;transition:opacity .5s;}}
#app.on{{opacity:1;}}
#sb{{width:254px;flex-shrink:0;background:var(--s);border-right:1px solid var(--b1);display:flex;flex-direction:column;overflow:hidden;}}
.sbh{{padding:13px 15px;border-bottom:1px solid var(--b1);background:var(--c);display:flex;align-items:center;gap:10px;}}
.sblogo{{font-family:'Orbitron',monospace;font-size:15px;font-weight:900;color:var(--red);}}
.hbtn{{margin-left:auto;background:none;border:1px solid var(--b2);color:var(--mu);font-size:16px;width:30px;height:30px;border-radius:6px;cursor:pointer;display:flex;align-items:center;justify-content:center;transition:all .2s;}}
.hbtn:hover{{border-color:var(--red);color:var(--red);}}
.sbm{{display:grid;grid-template-columns:1fr 1fr;gap:5px;padding:9px 11px;border-bottom:1px solid var(--b1);}}
.sm{{background:var(--c);border:1px solid var(--b1);border-radius:6px;padding:7px 9px;text-align:center;}}
.smv{{font-family:'Orbitron',monospace;font-size:14px;font-weight:700;color:white;}}
.sml{{font-size:8px;color:var(--mu);letter-spacing:.07em;margin-top:2px;}}
.dlist{{flex:1;overflow-y:auto;}}
.dlist::-webkit-scrollbar{{width:2px;}}
.dlist::-webkit-scrollbar-thumb{{background:var(--b2);}}
.dbtn{{width:100%;background:none;border:none;border-left:3px solid transparent;padding:7px 11px;cursor:pointer;display:flex;align-items:center;gap:8px;transition:background .15s,border-color .15s;border-bottom:1px solid var(--b1);}}
.dbtn:hover{{background:#ffffff07;border-left-color:var(--tc);}}
.dbtn.active{{background:#ffffff12;border-left-color:var(--tc);}}
.dr{{font-family:'Orbitron',monospace;font-size:9px;color:var(--mu);width:16px;flex-shrink:0;}}
.dav{{width:33px;height:33px;border-radius:50%;border:1.5px solid;overflow:hidden;flex-shrink:0;background:var(--c);display:flex;align-items:center;justify-content:center;}}
.dav img{{width:100%;height:100%;object-fit:cover;object-position:top;}}
.dfb{{font-size:9px;font-weight:700;}}
.di{{flex:1;min-width:0;text-align:left;}}
.dt{{display:flex;align-items:center;gap:5px;margin-bottom:1px;}}
.da{{font-family:'Orbitron',monospace;font-size:11px;font-weight:700;}}
.dl{{height:13px;border-radius:2px;}}
.dn{{font-size:10px;color:var(--su);white-space:nowrap;overflow:hidden;text-overflow:ellipsis;display:block;}}
.dp{{font-family:'Orbitron',monospace;font-size:11px;color:var(--mu);font-weight:600;}}
#main{{flex:1;overflow-y:auto;scroll-behavior:smooth;}}
#main::-webkit-scrollbar{{width:3px;}}
#main::-webkit-scrollbar-thumb{{background:var(--b2);border-radius:3px;}}
#welcome{{padding:40px 48px;}}
.wt{{font-family:'Orbitron',monospace;font-size:11px;color:var(--mu);letter-spacing:.2em;margin-bottom:6px;}}
.wh{{font-family:'Orbitron',monospace;font-size:36px;font-weight:900;color:white;line-height:1.1;margin-bottom:5px;}}
.wh span{{color:var(--red);}}
.ws{{font-size:14px;color:var(--su);margin-bottom:28px;}}
.wgrid{{display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin-bottom:22px;}}
.wstat{{background:var(--c);border:1px solid var(--b1);border-top:2px solid var(--red);border-radius:10px;padding:16px;}}
.wsv{{font-family:'Orbitron',monospace;font-size:24px;font-weight:900;color:var(--red);}}
.wsl{{font-size:9px;color:var(--mu);letter-spacing:.1em;margin-top:4px;}}
.cc{{background:var(--c);border:1px solid var(--b1);border-radius:11px;padding:18px;margin-bottom:16px;}}
.clbl{{font-family:'Orbitron',monospace;font-size:9px;color:var(--mu);letter-spacing:.14em;margin-bottom:12px;}}
.tw2{{display:grid;grid-template-columns:1fr 1fr;gap:14px;}}
.fi-r{{display:flex;align-items:center;gap:9px;margin-bottom:9px;}}
.fi-l{{font-size:12px;color:var(--su);width:116px;flex-shrink:0;}}
.fi-t{{flex:1;height:4px;background:var(--b2);border-radius:2px;overflow:hidden;}}
.fi-f{{height:100%;border-radius:2px;}}
.ar{{display:flex;align-items:center;padding:5px 0;border-bottom:1px solid var(--b1);gap:8px;}}
.ar:last-child{{border:none;}}
.ad{{font-family:'Orbitron',monospace;font-size:10px;font-weight:700;width:34px;}}
.ab{{flex:1;display:flex;flex-direction:column;gap:3px;}}
.abw{{height:3px;background:var(--b2);border-radius:2px;overflow:hidden;}}
.af{{height:100%;border-radius:2px;}}
.dpanel{{display:none;opacity:0;transform:translateY(12px);transition:opacity .35s ease,transform .35s ease;}}
.dpanel.shown{{opacity:1;transform:none;}}
.hero{{position:relative;display:flex;align-items:flex-end;min-height:240px;overflow:hidden;background:linear-gradient(105deg,#03030c 0%,#070715 40%,#03030c 100%);border-bottom:1px solid var(--b1);}}
.hero::before{{content:'';position:absolute;inset:0;pointer-events:none;background:radial-gradient(ellipse 75% 110% at 90% 50%,var(--tc) 0%,transparent 60%);opacity:.14;}}
.hero::after{{content:'';position:absolute;inset:0;pointer-events:none;background:radial-gradient(ellipse 35% 70% at 100% 100%,var(--tc) 0%,transparent 55%);opacity:.09;}}
.hstripe{{position:absolute;inset:0;background:repeating-linear-gradient(-55deg,transparent,transparent 36px,var(--tc) 36px,var(--tc) 37px);opacity:.05;}}
.hghost{{position:absolute;right:20px;top:8px;font-family:'Orbitron',monospace;font-size:108px;font-weight:900;color:white;opacity:.04;line-height:1;pointer-events:none;}}
.hphoto-col{{flex-shrink:0;width:200px;align-self:flex-end;}}
.hframe{{position:relative;width:182px;height:218px;margin-left:26px;border-top:3px solid;border-left:3px solid;border-right:3px solid;border-radius:8px 8px 0 0;overflow:hidden;background:var(--c);display:flex;align-items:center;justify-content:center;}}
.himg{{width:100%;height:100%;object-fit:cover;object-position:center top;display:block;}}
.hfallback{{font-family:'Orbitron',monospace;font-size:28px;font-weight:900;padding:20px;position:absolute;inset:0;display:flex;align-items:center;justify-content:center;}}
.hglow{{position:absolute;inset:0;pointer-events:none;}}
.hinfo{{flex:1;padding:28px 24px 26px;display:flex;flex-direction:column;gap:9px;}}
.hliv-row{{display:flex;align-items:center;gap:10px;}}
.hliv{{height:20px;border-radius:3px;}}
.hnb{{font-family:'Orbitron',monospace;font-size:12px;font-weight:700;border:1px solid;border-radius:4px;padding:3px 9px;letter-spacing:.08em;}}
.hname{{font-family:'Orbitron',monospace;font-size:32px;font-weight:900;color:white;letter-spacing:.04em;line-height:1.1;}}
.hteam{{font-family:'Orbitron',monospace;font-size:11px;font-weight:600;letter-spacing:.14em;}}
.hmodels{{display:flex;gap:10px;}}
.hmb{{display:flex;flex-direction:column;align-items:center;gap:2px;border:1px solid;border-radius:8px;padding:9px 14px;min-width:106px;}}
.hmv{{font-family:'Orbitron',monospace;font-size:24px;font-weight:900;line-height:1;}}
.hml{{font-size:8px;color:var(--mu);letter-spacing:.1em;}}
.hero-car-wrap{{position:relative;align-self:center;margin-right:20px;flex-shrink:0;}}
.hero-car{{width:260px;height:auto;display:block;filter:drop-shadow(0 4px 16px rgba(0,0,0,.5));animation:carFloat 3s ease-in-out infinite;}}
@keyframes carFloat{{0%,100%{{transform:translateY(0px) rotate(-1deg);}}50%{{transform:translateY(-6px) rotate(1deg);}}}}
.sstrip{{display:grid;grid-template-columns:repeat(10,1fr);border-bottom:1px solid var(--b1);}}
.st{{padding:11px 5px;text-align:center;border-right:1px solid var(--b1);position:relative;overflow:hidden;}}
.st::after{{content:'';position:absolute;bottom:0;left:0;right:0;height:2px;background:var(--tc);opacity:.35;}}
.st:last-child{{border-right:none;}}
.sv{{font-family:'Orbitron',monospace;font-size:16px;font-weight:700;}}
.sl{{font-size:8px;color:var(--mu);letter-spacing:.06em;margin-top:4px;line-height:1.3;}}
.pad{{padding:14px 20px 28px;position:relative;}}
.pad::before{{content:'';position:absolute;top:0;left:0;right:0;height:180px;background:radial-gradient(ellipse 60% 100% at 50% 0%,var(--tc) 0%,transparent 70%);opacity:.06;pointer-events:none;z-index:0;}}
.pad>*{{position:relative;z-index:1;}}
.sumcard{{background:var(--c);border:1px solid var(--b1);border-radius:10px;padding:18px 22px;border-top:2px solid;margin-bottom:13px;}}
.sumtxt{{font-size:14px;color:#a0a0c8;line-height:1.85;margin-top:10px;letter-spacing:.02em;}}
.charts2{{display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-bottom:13px;}}
.ccard{{background:var(--c);border:1px solid var(--b1);border-radius:10px;padding:14px 16px;margin-bottom:13px;}}
.strat-scroll{{display:flex;gap:8px;overflow-x:auto;padding-bottom:8px;}}
.strat-scroll::-webkit-scrollbar{{height:3px;}}
.strat-scroll::-webkit-scrollbar-thumb{{background:var(--b2);border-radius:3px;}}
.sc{{flex-shrink:0;width:120px;background:var(--s);border:1px solid var(--b2);border-radius:10px;padding:10px 11px;display:flex;flex-direction:column;gap:6px;transition:border-color .15s,transform .15s;cursor:default;}}
.sc:hover{{border-color:var(--tc);transform:translateY(-2px);}}
.sc-rnd{{font-family:'Orbitron',monospace;font-size:9px;color:var(--mu);font-weight:700;}}
.sc-loc{{font-size:11px;color:white;font-weight:600;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;}}
.sc-pos{{padding:4px 0;border-radius:6px;font-weight:700;font-size:16px;font-family:'Orbitron',monospace;text-align:center;}}
.sc-gain{{font-size:12px;font-weight:700;font-family:'Orbitron',monospace;text-align:center;}}
.sc-stops{{font-size:9px;color:var(--mu);font-family:'Orbitron',monospace;text-align:center;}}
.sc-bot{{display:flex;align-items:center;justify-content:space-between;border-top:1px solid var(--b2);padding-top:5px;margin-top:2px;}}
.sc-pts{{font-family:'Orbitron',monospace;font-size:12px;font-weight:700;color:white;}}
.sc-cumpts{{font-family:'Orbitron',monospace;font-size:9px;color:var(--mu);}}
.twrap{{overflow-x:auto;border:1px solid var(--b1);border-radius:9px;}}
.rtbl{{width:100%;border-collapse:collapse;font-size:12px;}}
.rtbl th{{padding:8px 10px;text-align:left;font-family:'Orbitron',monospace;font-size:8px;font-weight:600;color:var(--mu);letter-spacing:.1em;background:var(--c);white-space:nowrap;}}
.thrf{{color:var(--gold)!important;}}
.thnn{{color:#a78bfa!important;}}
.rtbl td{{padding:8px 10px;border-bottom:1px solid var(--b1);}}
.tr:hover td{{background:#ffffff04;}}
.tm{{color:var(--su);}}
.tw{{color:white;}}
.tg{{color:var(--gold);font-weight:700;font-family:'Orbitron',monospace;font-size:11px;}}
.pb{{display:inline-block;padding:2px 7px;border-radius:5px;font-weight:700;font-size:11px;font-family:'Orbitron',monospace;white-space:nowrap;}}
.tp{{padding:2px 6px;border-radius:4px;font-size:9px;font-weight:600;border:1px solid;white-space:nowrap;font-family:'Orbitron',monospace;}}
</style>
</head>
<body>

<div id="intro">
  <div class="igrid"></div>
  <div class="isl" style="top:27%;left:0;width:62%;animation-delay:.35s;"></div>
  <div class="isl" style="top:40%;left:0;width:80%;animation-delay:.5s;"></div>
  <div class="isl" style="top:52%;left:0;width:70%;animation-delay:.42s;"></div>
  <div class="isl" style="top:63%;left:0;width:52%;animation-delay:.6s;"></div>
  <div class="iblur"></div>
  <div class="icar" id="icar">
    <svg width="210" height="70" viewBox="0 0 300 80" fill="none">
      <ellipse cx="14" cy="46" rx="16" ry="5" fill="#FF4500" opacity=".9"><animate attributeName="rx" values="16;22;13;19;16" dur=".13s" repeatCount="indefinite"/></ellipse>
      <ellipse cx="9" cy="46" rx="9" ry="3" fill="#FFD700"><animate attributeName="rx" values="9;14;7;11;9" dur=".11s" repeatCount="indefinite"/></ellipse>
      <rect x="20" y="16" width="32" height="5" rx="2.5" fill="#e94560"/>
      <rect x="29" y="21" width="3" height="14" fill="#e94560"/>
      <rect x="40" y="21" width="3" height="14" fill="#e94560"/>
      <path d="M26 52 L52 33 L232 33 L264 52 Z" fill="#CC1A1A"/>
      <path d="M62 33 L84 22 L202 22 L224 33 Z" fill="#e94560"/>
      <path d="M112 22 L128 12 L174 12 L190 22 Z" fill="#111"/>
      <path d="M122 22 L134 15 L168 15 L180 22 Z" fill="#1a1a3a" opacity=".8"/>
      <path d="M228 41 L274 44 L275 49 L224 50 Z" fill="#CC1A1A"/>
      <path d="M262 33 L288 43 L288 50 L260 50 Z" fill="#e94560"/>
      <rect x="268" y="49" width="28" height="4" rx="2" fill="#e94560"/>
      <circle cx="68" cy="54" r="15" fill="#1a1a1a"/><circle cx="68" cy="54" r="10" fill="#2a2a2a"/><circle cx="68" cy="54" r="5" fill="#3a3a3a"/><circle cx="68" cy="54" r="2" fill="#555"/>
      <circle cx="242" cy="54" r="14" fill="#1a1a1a"/><circle cx="242" cy="54" r="9.5" fill="#2a2a2a"/><circle cx="242" cy="54" r="4.5" fill="#3a3a3a"/><circle cx="242" cy="54" r="2" fill="#555"/>
    </svg>
  </div>
  <div class="itxt">
    <div class="ilogo">F1 <span>2025</span></div>
    <div class="isub">SEASON INTELLIGENCE DASHBOARD</div>
    <div class="iprog"><div class="ibar"></div></div>
    <button class="icta" onclick="enterPaddock()">&#9654;&nbsp; ENTER PADDOCK</button>
  </div>
</div>

<div id="app">
  <div id="sb">
    <div class="sbh">
      <div class="sblogo">&#9654; F1 2025</div>
      <button class="hbtn" onclick="goHome()" title="Back to home">&#8962;</button>
    </div>
    <div class="sbm">
      <div class="sm"><div class="smv" style="color:#2ecc71;">{rf_acc}%</div><div class="sml">RF \xb12 ACC</div></div>
      <div class="sm"><div class="smv" style="color:#a78bfa;">{nn_acc}%</div><div class="sml">NN \xb12 ACC</div></div>
      <div class="sm"><div class="smv">{rf_mae}</div><div class="sml">RF MAE</div></div>
      <div class="sm"><div class="smv">{nn_mae}</div><div class="sml">NN MAE</div></div>
    </div>
    <div class="dlist">{sidebar}</div>
  </div>
  <div id="main">
    <div id="welcome">
      <div class="wt">FORMULA ONE \xb7 2025 SEASON</div>
      <div class="wh">MACHINE LEARNING<br><span>RACE PREDICTOR</span></div>
      <div class="ws">Random Forest + Neural Network \xb7 26,692 laps \xb7 24 races \xb7 21 drivers \u2014 select a driver</div>
      <div class="wgrid">
        <div class="wstat"><div class="wsv">{rf_acc}%</div><div class="wsl">RANDOM FOREST \xb12 ACCURACY</div></div>
        <div class="wstat"><div class="wsv" style="color:#a78bfa;">{nn_acc}%</div><div class="wsl">NEURAL NETWORK \xb12 ACCURACY</div></div>
        <div class="wstat"><div class="wsv">{rf_mae}</div><div class="wsl">RF AVG ERROR (positions)</div></div>
        <div class="wstat"><div class="wsv">{nn_mae}</div><div class="wsl">NN AVG ERROR (positions)</div></div>
      </div>
      <div class="cc">
        <div class="clbl">CHAMPIONSHIP BATTLE \u2014 CUMULATIVE POINTS TOP 8</div>
        <div id="champ" style="height:280px;"></div>
      </div>
      <div class="tw2">
        <div class="cc" style="margin-bottom:0;">
          <div class="clbl">FEATURE IMPORTANCE \u2014 WHAT WINS RACES</div>
          {fi_html}
        </div>
        <div class="cc" style="margin-bottom:0;">
          <div class="clbl">NN ACCURACY BY DRIVER (\xb12 pos \xb7 dim bar = RF, bright = NN)</div>
          {acc_html}
        </div>
      </div>
    </div>
    {panels}
  </div>
</div>

<script>
function playRev(done) {{
  try {{
    var ctx = new (window.AudioContext||window.webkitAudioContext)();
    var t   = ctx.currentTime;
    function makeDistortion(amount) {{
      var curve = new Float32Array(256);
      for (var i = 0; i < 256; i++) {{
        var x = (i * 2) / 256 - 1;
        curve[i] = (Math.PI + amount) * x / (Math.PI + amount * Math.abs(x));
      }}
      return curve;
    }}
    var dist = ctx.createWaveShaper();
    dist.curve = makeDistortion(300);
    dist.oversample = '4x';
    dist.connect(ctx.destination);
    [[55,0,2.8,0.28,8.0],[110,0,2.7,0.22,7.5],[220,0.02,2.5,0.18,6.5],[440,0.04,2.3,0.13,5.5],[880,0.06,2.0,0.08,4.5]].forEach(function(p) {{
      var o = ctx.createOscillator(), g = ctx.createGain(), d2 = ctx.createWaveShaper();
      d2.curve = makeDistortion(150); d2.oversample = '2x';
      o.connect(d2); d2.connect(g); g.connect(dist);
      o.type = 'sawtooth';
      o.frequency.setValueAtTime(p[0], t+p[1]);
      o.frequency.exponentialRampToValueAtTime(p[0]*p[4], t+p[1]+p[2]*0.55);
      o.frequency.exponentialRampToValueAtTime(p[0]*p[4]*0.6, t+p[1]+p[2]);
      g.gain.setValueAtTime(0, t+p[1]);
      g.gain.linearRampToValueAtTime(p[3], t+p[1]+0.04);
      g.gain.setValueAtTime(p[3], t+p[1]+p[2]-0.15);
      g.gain.exponentialRampToValueAtTime(0.001, t+p[1]+p[2]);
      o.start(t+p[1]); o.stop(t+p[1]+p[2]+0.05);
    }});
    var tw=ctx.createOscillator(), tg=ctx.createGain();
    tw.connect(tg); tg.connect(ctx.destination); tw.type='sine';
    tw.frequency.setValueAtTime(900,t+0.1);
    tw.frequency.exponentialRampToValueAtTime(8000,t+1.8);
    tw.frequency.exponentialRampToValueAtTime(400,t+2.9);
    tg.gain.setValueAtTime(0,t+0.1); tg.gain.linearRampToValueAtTime(0.12,t+0.3);
    tg.gain.setValueAtTime(0.12,t+1.8); tg.gain.exponentialRampToValueAtTime(0.001,t+3.0);
    tw.start(t+0.1); tw.stop(t+3.1);
    var bufSize=ctx.sampleRate*0.08, buf=ctx.createBuffer(1,bufSize,ctx.sampleRate);
    var data=buf.getChannelData(0);
    for(var i=0;i<bufSize;i++) data[i]=(Math.random()*2-1);
    var noise=ctx.createBufferSource(), ng=ctx.createGain(), nf=ctx.createBiquadFilter();
    nf.type='bandpass'; nf.frequency.value=1200; nf.Q.value=0.8;
    noise.buffer=buf; noise.connect(nf); nf.connect(ng); ng.connect(ctx.destination);
    ng.gain.setValueAtTime(0.35,t+1.85); ng.gain.exponentialRampToValueAtTime(0.001,t+1.93);
    noise.start(t+1.85); noise.stop(t+1.94);
    var kick=ctx.createOscillator(), kg=ctx.createGain();
    kick.connect(kg); kg.connect(ctx.destination); kick.type='sine';
    kick.frequency.setValueAtTime(180,t); kick.frequency.exponentialRampToValueAtTime(40,t+0.25);
    kg.gain.setValueAtTime(0.6,t); kg.gain.exponentialRampToValueAtTime(0.001,t+0.3);
    kick.start(t); kick.stop(t+0.31);
    if(done) setTimeout(done,3200);
  }} catch(e) {{ if(done) done(); }}
}}

function enterPaddock() {{ playRev(); doEnter(); }}

function doEnter() {{
  var intro=document.getElementById('intro'), app=document.getElementById('app');
  intro.style.transition='opacity .6s'; intro.style.opacity='0';
  setTimeout(function() {{ intro.style.display='none'; app.classList.add('on'); buildChamp(); }}, 650);
}}

setTimeout(function() {{
  if(document.getElementById('intro').style.display!=='none') doEnter();
}}, 4400);

function goHome() {{
  document.querySelectorAll('.dpanel').forEach(function(p) {{ p.style.display='none'; p.classList.remove('shown'); }});
  document.querySelectorAll('.dbtn').forEach(function(b) {{ b.classList.remove('active'); }});
  document.getElementById('welcome').style.display='block';
  document.getElementById('main').scrollTop=0;
}}

function go(drv) {{
  document.querySelectorAll('.dpanel').forEach(function(p) {{ p.style.display='none'; p.classList.remove('shown'); }});
  document.querySelectorAll('.dbtn').forEach(function(b) {{ b.classList.remove('active'); }});
  document.getElementById('welcome').style.display='none';
  var panel=document.getElementById('panel-'+drv);
  panel.style.display='block';
  document.getElementById('btn-'+drv).classList.add('active');
  document.getElementById('main').scrollTop=0;
  requestAnimationFrame(function() {{ requestAnimationFrame(function() {{ panel.classList.add('shown'); }}); }});
}}

function buildChamp() {{
  var s={champ_data}, rl={rl_data}, traces=[];
  for(var d in s) {{
    var x=s[d];
    traces.push({{x:x.r,y:x.c,mode:'lines+markers',name:d,
      line:{{color:x.col,width:2.5}},marker:{{size:5,color:x.col}},
      hovertemplate:'<b>'+d+'</b> R%{{x}}: %{{y}} pts<extra></extra>'}});
  }}
  Plotly.newPlot('champ',traces,{{
    paper_bgcolor:'transparent',plot_bgcolor:'transparent',
    font:{{color:'#606080',family:'Orbitron,monospace',size:10}},
    margin:{{l:36,r:10,t:6,b:48}},
    xaxis:{{gridcolor:'#181830',tickvals:[1,4,8,12,16,20,24],
            ticktext:['AUS','BAH','MON','GBR','ITA','USA','ABU'],tickfont:{{size:9}}}},
    yaxis:{{gridcolor:'#181830',title:'Points'}},
    legend:{{bgcolor:'transparent',orientation:'h',y:-0.3}},
    hovermode:'x unified',hoverlabel:{{bgcolor:'#050518',font:{{color:'white'}}}}
  }},{{responsive:true,displayModeBar:false}});
}}

document.body.style.cursor="url(data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSIwIDAgNDAgMTgiIHhtbG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGVsbGlwc2UgY3g9IjEwIiBjeT0iMTUiIHJ4PSI1IiByeT0iNCIgZmlsbD0iIzFhMWExYSIvPgogIDxlbGxpcHNlIGN4PSIxMCIgY3k9IjE1IiByeD0iMy41IiByeT0iMi44IiBmaWxsPSIjMzMzIi8+CiAgPGVsbGlwc2UgY3g9IjMwIiBjeT0iMTUiIHJ4PSI1IiByeT0iNCIgZmlsbD0iIzFhMWExYSIvPgogIDxlbGxpcHNlIGN4PSIzMCIgY3k9IjE1IiByeD0iMy41IiByeT0iMi44IiBmaWxsPSIjMzMzIi8+CiAgPHBhdGggZD0iTTYgMTIgTDggNiBMMzIgNiBMMzQgMTIgTDMyIDE0IEw4IDE0IFoiIGZpbGw9IiNlOTQ1NjAiLz4KICA8cGF0aCBkPSJNMTAgNiBMMTIgMyBMMjggMyBMMzAgNiBaIiBmaWxsPSIjY2MxYTNhIi8+CiAgPHJlY3QgeD0iMiIgeT0iNiIgd2lkdGg9IjUiIGhlaWdodD0iMyIgcng9IjEiIGZpbGw9IiNlOTQ1NjAiIG9wYWNpdHk9Ii43Ii8+CiAgPHJlY3QgeD0iMzMiIHk9IjUiIHdpZHRoPSI2IiBoZWlnaHQ9IjIiIHJ4PSIxIiBmaWxsPSIjZTk0NTYwIi8+CiAgPHJlY3QgeD0iMzYiIHk9IjciIHdpZHRoPSIzIiBoZWlnaHQ9IjEiIHJ4PSIuNSIgZmlsbD0iI2U5NDU2MCIgb3BhY2l0eT0iLjUiLz4KICA8cmVjdCB4PSIxNCIgeT0iNyIgd2lkdGg9IjEyIiBoZWlnaHQ9IjQiIHJ4PSIxIiBmaWxsPSIjMDgwODEyIi8+CiAgPHBhdGggZD0iTTE2IDggUTIwIDUgMjQgOCIgc3Ryb2tlPSIjZmZmZmZmIiBzdHJva2Utd2lkdGg9IjEuNSIgZmlsbD0ibm9uZSIgb3BhY2l0eT0iLjYiLz4KPC9zdmc+) 4 4, crosshair";
</script>
</body>
</html>"""

with open('f1_dashboard.html', 'w', encoding='utf-8') as f:
    f.write(HTML)

print(f"\n\u2705  Dashboard saved!")
print(f"    RF  \xb12: {rf_acc}%  |  MAE: {rf_mae}")
print(f"    NN  \xb12: {nn_acc}%  |  MAE: {nn_mae}")
print(f"    Pit stops: {len(pit_df)}")

from google.colab import files
files.download('f1_dashboard.html')
print("\u2705  Downloading \u2014 open in Chrome!")

⚙️  Training Neural Network...
   ✅ NN done — ±2 accuracy: 33.0%  MAE: 3.99
⚙️  Processing pit stops...


In [ ]:
import subprocess
from IPython.display import HTML

# Display inline in Colab
with open('f1_dashboard.html', 'r') as f:
    content = f.read()

display(HTML(f'<iframe srcdoc="{content.replace(chr(34), chr(39))}" width="100%" height="700px" style="border:none; border-radius:12px;"></iframe>'))